# Step 2: Heckman Control Function + DR-AIPW

Estima el efecto causal de **X1** (tratamiento sintético) sobre **Y** (default).

Comparación:
- **Baseline**: solo `accepted_labeled.csv` (muestra sesgada)
- **BASL-expanded**: `D_tilde.csv` (población recuperada)

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.special import ndtr  # Phi(x)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## 1. Cargar datos y construir tratamiento sintético

Como no hay variable de homeownership real, construimos:
- **T** (tratamiento): binarización de X1 (umbral = mediana)
- **Z** (instrumento/exclusión): variable sintética correlacionada con T pero no con Y directamente

In [2]:
def load_and_prepare(path, treatment_col="X1", target_col="y", seed=42):
    """
    Carga el dataset y construye:
    - T: tratamiento binario (X1 >= mediana)
    - Z: instrumento sintético (correlacionado con X1, independiente de Y|X)
    - Y: outcome (default)
    - X: covariables de control (X2, X3, X4)
    """
    df = pd.read_csv(path)
    # Quitar columnas administrativas si existen
    drop_cols = [c for c in ["split", "Unnamed: 0"] if c in df.columns]
    df = df.drop(columns=drop_cols)

    rng = np.random.default_rng(seed)

    # Tratamiento: X1 >= mediana → T=1 ("homeowner")
    median_x1 = df[treatment_col].median()
    df["T"] = (df[treatment_col] >= median_x1).astype(int)

    # Instrumento Z: correlacionado con X1 pero no con Y|X
    # Simula una variable regional (ej. acceso a programa de subsidio)
    df["Z"] = 0.6 * df[treatment_col] + rng.normal(0, 0.8, len(df))

    feature_cols = ["X2", "X3", "X4"]  # covariables de control
    Y = df[target_col].values
    T = df["T"].values
    Z = df["Z"].values
    X = df[feature_cols].values

    print(f"N = {len(df):,}")
    print(f"Bad rate: {Y.mean():.1%}")
    print(f"Treatment rate (T=1): {T.mean():.1%}")
    print(f"Corr(Z, T): {np.corrcoef(Z, T)[0,1]:.3f}")
    print(f"Corr(Z, Y): {np.corrcoef(Z, Y)[0,1]:.3f}  (debe ser bajo)")

    return Y, T, X, Z, feature_cols


print("=== D_tilde (BASL-expanded) ===")
Y_exp, T_exp, X_exp, Z_exp, feat_cols = load_and_prepare("data/D_tilde.csv")

print("\n=== Accepted only (baseline) ===")
Y_acc, T_acc, X_acc, Z_acc, _ = load_and_prepare("data/accepted_labeled.csv")

=== D_tilde (BASL-expanded) ===
N = 10,700
Bad rate: 41.3%
Treatment rate (T=1): 50.0%
Corr(Z, T): 0.495
Corr(Z, Y): 0.172  (debe ser bajo)

=== Accepted only (baseline) ===
N = 14,016
Bad rate: 34.5%
Treatment rate (T=1): 50.0%
Corr(Z, T): 0.472
Corr(Z, Y): 0.073  (debe ser bajo)


## 2. Heckman Control Function

**First stage**: probit de T sobre (X, Z)  
**Control function**: inverse Mills ratio λ = φ(ν̂) / Φ(ν̂)  
λ captura la parte de la asignación de tratamiento explicada por factores latentes no observados.

In [3]:
def heckman_control_function(T, X, Z, scaler=None):
    """
    Estima el probit de primera etapa y retorna:
    - lambda_hat: inverse Mills ratio (control function)
    - nu_hat: índice lineal del probit
    - probit_model: modelo ajustado
    """
    # Construir matriz de features para el probit: [X, Z]
    XZ = np.column_stack([X, Z.reshape(-1, 1)])

    if scaler is None:
        scaler = StandardScaler()
        XZ_scaled = scaler.fit_transform(XZ)
    else:
        XZ_scaled = scaler.transform(XZ)

    # Probit via logistic regression (aproximación)
    # Para probit exacto multiplicamos coefs por π/√3
    probit = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    probit.fit(XZ_scaled, T)

    # Índice lineal ν̂
    nu_hat = XZ_scaled @ probit.coef_[0] + probit.intercept_[0]
    # Ajuste logit → probit: multiplicar por π/√3 ≈ 1.814 (no es estrictamente
    # necesario para el control function, pero mejora la escala)

    # Inverse Mills ratio: λ = φ(ν̂) / Φ(ν̂)
    phi_nu  = norm.pdf(nu_hat)          # densidad normal estándar
    Phi_nu  = norm.cdf(nu_hat)          # CDF normal estándar
    lambda_hat = phi_nu / np.clip(Phi_nu, 1e-8, 1 - 1e-8)

    # First-stage AUC
    T_pred = probit.predict_proba(XZ_scaled)[:, 1]
    auc_fs = roc_auc_score(T, T_pred)
    print(f"First stage AUC (probit T~X+Z): {auc_fs:.4f}")
    print(f"λ media: {lambda_hat.mean():.4f}, std: {lambda_hat.std():.4f}")

    return lambda_hat, nu_hat, probit, scaler


print("--- BASL-expanded ---")
lambda_exp, nu_exp, probit_exp, scaler_exp = heckman_control_function(T_exp, X_exp, Z_exp)

print("\n--- Accepted only ---")
lambda_acc, nu_acc, probit_acc, scaler_acc = heckman_control_function(T_acc, X_acc, Z_acc)

--- BASL-expanded ---
First stage AUC (probit T~X+Z): 0.7904
λ media: 0.9780, std: 0.8078

--- Accepted only ---
First stage AUC (probit T~X+Z): 0.7752
λ media: 0.9524, std: 0.7446


## 3. DR-AIPW con cross-fitting

Estima el PATE usando el espacio de covariables aumentado **(X, λ)**.

$$\hat{\tau}^{DR} = \frac{1}{n}\sum_i \left[ \frac{T_i(Y_i - \hat{\mu}_1)}{\hat{e}} - \frac{(1-T_i)(Y_i - \hat{\mu}_0)}{1-\hat{e}} + \hat{\mu}_1 - \hat{\mu}_0 \right]$$

Con **5-fold cross-fitting** para evitar overfitting en la estimación de los nuisance models.

In [4]:
def dr_aipw_crossfit(Y, T, X, lambda_hat, n_folds=5, seed=42):
    """
    DR-AIPW con cross-fitting.

    Nuisance models:
    - e(X,λ)     : propensity score → GradientBoostingClassifier
    - μ_t(X,λ)   : outcome model    → GradientBoostingClassifier

    Returns:
    - tau_hat : estimación puntual del PATE
    - se      : standard error (bootstrap)
    - scores  : vector de scores DR individuales (para bootstrap)
    """
    # Espacio aumentado: (X, λ)
    W = np.column_stack([X, lambda_hat.reshape(-1, 1)])

    n = len(Y)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)

    # Arrays para guardar predicciones out-of-fold
    e_hat  = np.zeros(n)   # propensity score
    mu1_hat = np.zeros(n)  # E[Y|T=1, X, λ]
    mu0_hat = np.zeros(n)  # E[Y|T=0, X, λ]

    for fold, (train_idx, val_idx) in enumerate(kf.split(W)):
        W_tr, W_val   = W[train_idx], W[val_idx]
        Y_tr, Y_val   = Y[train_idx], Y[val_idx]
        T_tr, T_val   = T[train_idx], T[val_idx]

        # Propensity score
        ps_model = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1, random_state=seed
        )
        ps_model.fit(W_tr, T_tr)
        e_hat[val_idx] = np.clip(ps_model.predict_proba(W_val)[:, 1], 0.01, 0.99)

        # Outcome model T=1
        idx1 = train_idx[T_tr == 1]
        om1 = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1, random_state=seed
        )
        om1.fit(W[idx1], Y[idx1])
        mu1_hat[val_idx] = om1.predict_proba(W_val)[:, 1]

        # Outcome model T=0
        idx0 = train_idx[T_tr == 0]
        om0 = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1, random_state=seed
        )
        om0.fit(W[idx0], Y[idx0])
        mu0_hat[val_idx] = om0.predict_proba(W_val)[:, 1]

        print(f"  Fold {fold+1}: PS AUC={roc_auc_score(T_val, e_hat[val_idx]):.3f}")

    # DR-AIPW scores individuales
    scores = (
        T * (Y - mu1_hat) / e_hat
        - (1 - T) * (Y - mu0_hat) / (1 - e_hat)
        + mu1_hat - mu0_hat
    )

    tau_hat = scores.mean()

    # Standard error via bootstrap
    rng = np.random.default_rng(seed)
    boot_taus = []
    for _ in range(500):
        idx_b = rng.integers(0, n, size=n)
        boot_taus.append(scores[idx_b].mean())
    se = np.std(boot_taus)

    return tau_hat, se, scores


print("=" * 50)
print("DR-AIPW — BASL-expanded (D̃)")
print("=" * 50)
tau_exp, se_exp, scores_exp = dr_aipw_crossfit(Y_exp, T_exp, X_exp, lambda_exp)

print("\n" + "=" * 50)
print("DR-AIPW — Accepted only (baseline)")
print("=" * 50)
tau_acc, se_acc, scores_acc = dr_aipw_crossfit(Y_acc, T_acc, X_acc, lambda_acc)

DR-AIPW — BASL-expanded (D̃)
  Fold 1: PS AUC=0.794
  Fold 2: PS AUC=0.777
  Fold 3: PS AUC=0.789
  Fold 4: PS AUC=0.777
  Fold 5: PS AUC=0.801

DR-AIPW — Accepted only (baseline)
  Fold 1: PS AUC=0.755
  Fold 2: PS AUC=0.773
  Fold 3: PS AUC=0.773
  Fold 4: PS AUC=0.790
  Fold 5: PS AUC=0.767


## 4. Resultados y comparación

In [5]:
from scipy.stats import norm as normal

def report(label, tau, se):
    z    = tau / se
    pval = 2 * (1 - normal.cdf(abs(z)))
    ci_lo = tau - 1.96 * se
    ci_hi = tau + 1.96 * se
    sig   = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
    print(f"{label}")
    print(f"  τ̂ (PATE)   = {tau:+.4f} {sig}")
    print(f"  SE         = {se:.4f}")
    print(f"  95% CI     = [{ci_lo:+.4f}, {ci_hi:+.4f}]")
    print(f"  p-value    = {pval:.4f}")
    print()

print("=" * 55)
print("EFECTO CAUSAL DE X1 (tratamiento) SOBRE Y (default)")
print("Heckman Control Function + DR-AIPW")
print("=" * 55)
report("1. BASL-expanded (D̃) — población recuperada", tau_exp, se_exp)
report("2. Accepted only   — muestra sesgada",         tau_acc, se_acc)

diff = tau_exp - tau_acc
print("-" * 55)
print(f"Diferencia (D̃ - Accepted): {diff:+.4f}")
print("Si |diff| > 0: el sesgo de selección distorsionaba el efecto causal.")
print("=" * 55)

EFECTO CAUSAL DE X1 (tratamiento) SOBRE Y (default)
Heckman Control Function + DR-AIPW
1. BASL-expanded (D̃) — población recuperada
  τ̂ (PATE)   = +0.2423 ***
  SE         = 0.0118
  95% CI     = [+0.2191, +0.2655]
  p-value    = 0.0000

2. Accepted only   — muestra sesgada
  τ̂ (PATE)   = +0.0831 ***
  SE         = 0.0100
  95% CI     = [+0.0634, +0.1027]
  p-value    = 0.0000

-------------------------------------------------------
Diferencia (D̃ - Accepted): +0.1592
Si |diff| > 0: el sesgo de selección distorsionaba el efecto causal.


## Benchmarks


In [6]:
def dr_aipw_continuous_crossfit(Y, T, X, lambda_hat, n_folds=5, seed=42):
    """
    DR-AIPW para tratamiento CONTINUO (partially linear model).
    Estima E[dE[Y|T,X]/dT] via Robinson (1988) partialling-out.
    
    Nuisance models:
    - m(X,λ) = E[T|X,λ]  → GradientBoostingRegressor
    - g(X,λ) = E[Y|X,λ]  → GradientBoostingClassifier
    
    τ̂ = E[(T - m̂)(Y - ĝ)] / E[(T - m̂)²]
    """
    W = np.column_stack([X, lambda_hat.reshape(-1, 1)])
    n = len(Y)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)

    m_hat = np.zeros(n)   # E[T|X,λ]
    g_hat = np.zeros(n)   # E[Y|X,λ]

    for fold, (train_idx, val_idx) in enumerate(kf.split(W)):
        W_tr, W_val = W[train_idx], W[val_idx]
        Y_tr = Y[train_idx]
        T_tr = T[train_idx]

        # E[T|X,λ] — regresión porque T es continuo
        m_model = GradientBoostingRegressor(
            n_estimators=100, max_depth=3, learning_rate=0.1, random_state=seed
        )
        m_model.fit(W_tr, T_tr)
        m_hat[val_idx] = m_model.predict(W_val)

        # E[Y|X,λ] — clasificador porque Y es binario
        g_model = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1, random_state=seed
        )
        g_model.fit(W_tr, Y_tr)
        g_hat[val_idx] = g_model.predict_proba(W_val)[:, 1]

        T_res = T_tr - m_model.predict(W_tr)
        print(f"  Fold {fold+1}: R²(T~X)={1 - T_res.var()/T_tr.var():.3f}")

    # Robinson partialling-out
    T_res = T - m_hat
    Y_res = Y - g_hat
    tau_hat = (T_res * Y_res).mean() / (T_res ** 2).mean()

    # Bootstrap SE
    rng = np.random.default_rng(seed)
    boot_taus = []
    for _ in range(500):
        idx_b = rng.integers(0, n, size=n)
        T_res_b = T_res[idx_b]
        Y_res_b = Y_res[idx_b]
        boot_taus.append((T_res_b * Y_res_b).mean() / (T_res_b ** 2).mean())
    se = np.std(boot_taus)

    return tau_hat, se

In [7]:
# Estimar τ verdadero: efecto promedio de X1 sobre Y en la población oracle
# Como X1 e Y se generan de las mismas Gaussianas, τ_true ≈ correlación parcial
# Lo estimamos con DR-AIPW sobre el oracle completo con labels reales

df_oracle = pd.read_csv("data/oracle_population.csv")
Y_oracle = df_oracle["y"].values
T_oracle = df_oracle["X1"].values  # tratamiento continuo
X_oracle = df_oracle[["X2", "X3", "X4"]].values

In [8]:
tau_true, _ = dr_aipw_continuous_crossfit(
    Y_oracle, T_oracle, X_oracle, np.zeros(len(Y_oracle))
)
print(f"τ verdadero (oracle): {tau_true:.4f}")

  Fold 1: R²(T~X)=0.038
  Fold 2: R²(T~X)=0.040
  Fold 3: R²(T~X)=0.037
  Fold 4: R²(T~X)=0.041
  Fold 5: R²(T~X)=0.042
τ verdadero (oracle): 0.0837


## Benchmarks tables

In [9]:
# =============================================================================
# (i) NAIVE — DR-AIPW sobre D^a, sin BASL, sin Heckman
# =============================================================================
print("(i) NAIVE — DR-AIPW sobre D^a")
# Sin lambda (no Heckman)
tau_naive, se_naive = dr_aipw_continuous_crossfit(
    Y_acc, T_acc, X_acc,
    np.zeros(len(Y_acc))   # lambda = 0, sin corrección
)
print(f"τ̂ = {tau_naive:+.4f}, SE = {se_naive:.4f}")

(i) NAIVE — DR-AIPW sobre D^a
  Fold 1: R²(T~X)=0.037
  Fold 2: R²(T~X)=0.038
  Fold 3: R²(T~X)=0.038
  Fold 4: R²(T~X)=0.040
  Fold 5: R²(T~X)=0.040
τ̂ = +0.0913, SE = 0.0079


In [10]:
# =============================================================================
# (ii) BASL ONLY — logistic regression sobre D̃, sin Heckman
# =============================================================================
print("(ii) BASL ONLY — logistic regression sobre D̃")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
W_basl = scaler.fit_transform(
    np.column_stack([T_exp.reshape(-1,1), X_exp])
)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(W_basl, Y_exp)

# Efecto marginal de T: coef de X1 * mean(p*(1-p))
coef_T    = lr.coef_[0][0]
p_hat     = lr.predict_proba(W_basl)[:, 1]
tau_basl  = coef_T * (p_hat * (1 - p_hat)).mean()
# SE via bootstrap
rng = np.random.default_rng(42)
boot = []
for _ in range(500):
    idx_b = rng.integers(0, len(Y_exp), len(Y_exp))
    lr_b = LogisticRegression(max_iter=1000, random_state=42)
    lr_b.fit(W_basl[idx_b], Y_exp[idx_b])
    c_b = lr_b.coef_[0][0]
    p_b = lr_b.predict_proba(W_basl[idx_b])[:, 1]
    boot.append(c_b * (p_b * (1 - p_b)).mean())
se_basl = np.std(boot)
print(f"τ̂ = {tau_basl:+.4f}, SE = {se_basl:.4f}")

(ii) BASL ONLY — logistic regression sobre D̃
τ̂ = +0.1234, SE = 0.0040


In [11]:
# =============================================================================
# (iii) DR-AIPW ONLY — Heckman + DR-AIPW sobre D^a, sin BASL
# =============================================================================
print("(iii) DR-AIPW ONLY — Heckman + DR-AIPW sobre D^a")
# Heckman sobre D^a
lambda_acc_cf, _, _, _ = heckman_control_function(T_acc, X_acc, Z_acc)
# DR-AIPW con lambda sobre D^a
tau_draipw, se_draipw = dr_aipw_continuous_crossfit(
    Y_acc, T_acc, X_acc, lambda_acc_cf
)
print(f"τ̂ = {tau_draipw:+.4f}, SE = {se_draipw:.4f}")

(iii) DR-AIPW ONLY — Heckman + DR-AIPW sobre D^a
First stage AUC (probit T~X+Z): 0.7752
λ media: 0.9524, std: 0.7446
  Fold 1: R²(T~X)=0.269
  Fold 2: R²(T~X)=0.260
  Fold 3: R²(T~X)=0.263
  Fold 4: R²(T~X)=0.252
  Fold 5: R²(T~X)=0.266
τ̂ = +0.0751, SE = 0.0090


In [12]:
# =============================================================================
# TABLA COMPARATIVA
# =============================================================================
TAU_TRUE = 0.08 
print(f"\nτ verdadero (oracle): {TAU_TRUE:+.4f}\n")
print(f"{'Estimator':<40} {'τ̂':>8} {'SE':>8} {'95% CI':>22} {'Bias':>8}")
print("-" * 92)



estimators = [
    ("(i)  Naive",                        tau_naive,  se_naive),
    ("(ii) BASL only",                    tau_basl,   se_basl),
    ("(iii) DR-AIPW only",                tau_draipw, se_draipw),
    ("(iv) Ours (BASL+Heckman+DR-AIPW)",  tau_exp,    se_exp),
]

for name, tau, se in estimators:
    ci_lo = tau - 1.96 * se
    ci_hi = tau + 1.96 * se
    bias  = abs(tau - TAU_TRUE)
    print(f"{name:<40} {tau:>+8.4f} {se:>8.4f} "
          f"[{ci_lo:+.4f}, {ci_hi:+.4f}] {bias:>8.4f}")


τ verdadero (oracle): +0.0800

Estimator                                      τ̂       SE                 95% CI     Bias
--------------------------------------------------------------------------------------------
(i)  Naive                                +0.0913   0.0079 [+0.0757, +0.1068]   0.0113
(ii) BASL only                            +0.1234   0.0040 [+0.1156, +0.1311]   0.0434
(iii) DR-AIPW only                        +0.0751   0.0090 [+0.0576, +0.0927]   0.0049
(iv) Ours (BASL+Heckman+DR-AIPW)          +0.2423   0.0118 [+0.2191, +0.2655]   0.1623
